# CAM aerosol distributions in miam — shapes & box models

This tutorial chapter uses the configurations in [`cam_aerosol_configs.py`](../python/musica/examples/cam_aerosol_configs.py) to:

1. **Plot the size-distribution shape** of each CAM aerosol scheme
  - MAM (modal),
  - BAM (bulk),
  - CARMA (sectional).
2. Run a **short cloud-sulfate box model** for one representative size per scheme.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import musica
import musica.mechanism_configuration as mc
from musica.micm import MICM, SolverType, SolverState
from musica.mechanism_configuration import TwoMomentMode
from musica.examples.cam_aerosol_configs import (
    mam_representations,
    bam_representations,
    carma_representations,
    build_model,
)

## Part 1 — Size-distribution shapes

We'll create one panel per scheme, plotting the number distribution `dN/dln(r)` normalized to
unit total number, on a **shared log-radius axis (µm)** so the three schemes are directly comparable.

Two placement notes, since not every scheme prescribes an absolute size:

- **MAM (modal)**: `TwoMomentMode` fixes only σ_g — the modal diameter is *prognostic*, drifting
  within a fixed range. To place each mode on the shared axis we center it at the **geometric mean
  of its published diameter range** (Liu et al. 2012, *Geosci. Model Dev.* 5, 709–739, Table 1 —
  MAM3 rows for accumulation/aitken/coarse, the MAM7 row for primary carbon); σ_g is the real
  prescribed shape.
- **CARMA (sectional)**: the bins prescribe only edges, not a number-per-bin. We overlay an
  *illustrative* lognormal number distribution sampled onto the bins to show how a sectional scheme
  approximates a smooth distribution with piecewise-constant steps.

In [ ]:
def lognormal_dNdlnr(r, r_g, sigma_g):
    """Unit-area lognormal number distribution dN/dln(r)."""
    ln_s = np.log(sigma_g)
    return np.exp(-(np.log(r / r_g))**2 / (2.0 * ln_s**2)) / (np.sqrt(2.0 * np.pi) * ln_s)

radius = np.logspace(-9, -4, 600)         # absolute radius grid [m] (1 nm .. 100 um)
radius_um = radius * 1e6

# MAM modal diameters are prognostic within a fixed range; TwoMomentMode stores only
# sigma_g.  For placement on the shared axis we center each mode at the geometric-mean
# *radius* of its published diameter range -- Liu et al. (2012), Geosci. Model Dev. 5,
# 709-739, Table 1 (MAM3 rows for accum/aitken/coarse, MAM7 row for primary carbon):
#   mode            diameter range [um]   geometric-mean radius [m]
#   accumulation    0.058 - 0.27          sqrt(0.058*0.27)/2  = 6.26e-8
#   aitken          0.015 - 0.053         sqrt(0.015*0.053)/2 = 1.41e-8
#   coarse          0.80  - 3.65          sqrt(0.80*3.65)/2   = 8.54e-7
#   primary_carbon  0.039 - 0.13          sqrt(0.039*0.13)/2  = 3.56e-8
MAM_NOMINAL_RG = {
    "ACCUM": 6.26e-8, "AITKEN": 1.41e-8, "COARSE": 8.54e-7, "PRIMARY_CARBON": 3.56e-8,
}

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(8.5, 12))

# --- MAM4: modal.  TwoMomentMode fixes ONLY sigma_g (the diameter is diagnosed at
#     runtime within a range), so we place each mode at the geometric-mean radius of
#     its Liu et al. 2012 diameter range and plot the true sigma_g shape. ---
ax = axes[0]
for mode in mam_representations()["mam4"]:
    r_g = MAM_NOMINAL_RG[mode.name]
    ax.plot(radius_um, lognormal_dNdlnr(radius, r_g, mode.geometric_standard_deviation),
            label=f"{mode.name} (r_g≈{r_g*1e6:.3g} µm, σ={mode.geometric_standard_deviation})")
ax.set_xlabel("radius [µm]")
ax.set_title("MAM4 — modal (σ_g fixed; placed at geometric mean of Liu et al. 2012 range)")

# --- BAM: SingleMomentMode fixes both geometric mean radius and sigma_g ---
ax = axes[1]
for rep in bam_representations():
    ax.plot(radius_um, lognormal_dNdlnr(radius, rep.geometric_mean_radius,
                                        rep.geometric_standard_deviation),
            label=rep.name, lw=1.2)
ax.set_xlabel("radius [µm]")
ax.set_title("BAM — SingleMomentMode (fixed geometric mean radius + σ_g)")

# --- CARMA sulfate: UniformSection per bin, using each section's min/max radius.
#     The bins prescribe only edges, so we overlay an ILLUSTRATIVE lognormal number
#     distribution sampled onto the bins -> the sectional (piecewise-constant)
#     approximation to a smooth dN/dln(r). ---
ax = axes[2]
bins = carma_representations("sulfate")
edges = np.array([bins[0].min_radius] + [b.max_radius for b in bins]) * 1e6
centers = np.array([np.sqrt(b.min_radius * b.max_radius) for b in bins])   # geometric bin centers [m]
dlnr = np.log(edges[1:] / edges[:-1])
CARMA_RG, CARMA_SIGMA = 0.06e-6, 1.8          # illustrative background-sulfate lognormal
n_per_bin = lognormal_dNdlnr(centers, CARMA_RG, CARMA_SIGMA) * dlnr
n_per_bin /= n_per_bin.sum()                   # normalize to unit total N
heights = n_per_bin / dlnr                     # -> dN/dlnr
ax.stairs(heights, edges, fill=True, alpha=0.4, color="C0")
ax.stairs(heights, edges, color="C0", label=f"illustrative lognormal (r_g={CARMA_RG*1e6:.2g} µm)")
ax.set_title(f"CARMA sulfate — UniformSection per bin ({len(bins)} bins, illustrative N)")

for ax in axes:
    ax.set_xscale("log")
    ax.set_xlabel("radius [µm]")
    ax.set_ylabel("dN/dln(r)  (unit total N)")
    ax.legend(fontsize=7, ncol=2)
fig.tight_layout()
plt.show()

## Part 2 — Short cloud-sulfate box models (size-dependent uptake)

Reuse the S(IV)→S(VI) cloud-sulfate chemistry, now with **kinetic
`HenrysLawPhaseTransfer`** for the SO2/H2O2/O3 uptake
(`build_model(..., kinetic_uptake=True)`). The uptake rate scales with particle
**surface area (radius × number)**, so the three schemes' different size
distributions now drive different sulfate production. One representative each:

- **MAM4** — accumulation mode (`TwoMomentMode` → number set explicitly)
- **BAM** — sulfate (SO4) (`SingleMomentMode` → number derived from volume)
- **CARMA** — middle bin of the 30-bin sulfate group (`UniformSection`)

The initial state is **aerosol-scale** (small condensed water, ~1000 cm⁻³
number), *not* cloud LWC — with cloud water the sub-µm representations would get
an unphysically huge number and stiff dynamics.

In [ ]:
T_INIT, P_INIT = 280.0, 70000.0
GAS0_SO2, GAS0_H2O2, GAS0_O3 = 3.01e-8, 3.01e-8, 1.5e-6   # mol/m3  (~1 ppb SO2)
AEROSOL_H2O = 1e-7        # mol m-3 air   (~1.8 µg m-3 condensed water)
AEROSOL_NUMBER = 1e9      # #m-3 air     (~1000 cm-3; TwoMomentMode number IC)


def run_box(label, rep, target_time=1800.0):
    """Integrate the size-dependent (kinetic-uptake) cloud-sulfate chemistry."""
    mechanism = build_model(label, [rep], kinetic_uptake=True)
    micm = MICM(mechanism=mechanism,
                solver_type=SolverType.rosenbrock_dae4_standard_order,
                external_models=[musica.MIAM()])
    state = micm.create_state()
    mechanism.aerosol.set_default_parameters(state)      # radius / sigma params
    state.set_conditions(temperatures=T_INIT, pressures=P_INIT)

    p = f"{rep.name}.{rep.phases[0]}."                   # REPR.PHASE. prefix
    ics = {
        "SO2": GAS0_SO2, "H2O2": GAS0_H2O2, "O3": GAS0_O3,
        p + "H2O": AEROSOL_H2O,
        p + "SO2_aq": 1e-15, p + "H2O2_aq": 1e-15, p + "O3_aq": 1e-15,
        p + "Hp": 1e-6, p + "OHm": 1e-12, p + "HSO3m": 1e-15, p + "SO3mm": 1e-15,
        p + "SO4mm": 1e-9, p + "SO2OOHm": 0.0,
    }
    # TwoMomentMode (MAM) carries a prognostic number concentration -> set it;
    # SingleMomentMode/UniformSection derive number from the condensed volume.
    if isinstance(rep, TwoMomentMode):
        ics[f"{rep.name}.NUMBER_CONCENTRATION"] = AEROSOL_NUMBER
    state.set_concentrations(ics)

    t, dt = 0.0, 1e-4
    times, so4, ph = [], [], []
    while t < target_time - 1e-9:
        step = min(dt, target_time - t)
        result = micm.solve(state, time_step=step)
        assert result.state == SolverState.Converged, f"{label} failed at t={t:.4g}s"
        t += step
        for thresh in (0.01, 0.1, 1.0, 10.0, 100.0):    # ramp dt as the fast transient decays
            if t > thresh and dt < thresh:
                dt = thresh
        c = state.get_concentrations()
        times.append(t)
        so4.append(c[p + "SO4mm"][0])
        ph.append(-np.log10(max(c[p + "Hp"][0], 1e-30) / 1000.0))   # mol/m3 -> mol/L
    return times, so4, ph


representatives = {
    "MAM4 accum": mam_representations()["mam4"][0],
    "BAM SO4": bam_representations()[0],
    "CARMA sulfate (mid bin)": carma_representations("sulfate")[15],
}

In [ ]:
fig, (ax_so4, ax_ph) = plt.subplots(1, 2, figsize=(12, 4.5))
for label, rep in representatives.items():
    times, so4, ph = run_box(label, rep)
    ax_so4.plot(times, so4, label=label)
    ax_ph.plot(times, ph, label=label)

ax_so4.set_xlabel("time [s]"); ax_so4.set_ylabel("SO4²⁻ [mol m⁻³]")
ax_so4.set_title("Sulfate produced"); ax_so4.legend()
ax_ph.set_xlabel("time [s]"); ax_ph.set_ylabel("pH")
ax_ph.set_title("pH"); ax_ph.legend()
fig.tight_layout()
plt.show()